# NB08 — Clinical Severity Analysis (RQ4)

A pure CSV aggregation + statistics + plotting notebook. It groups pathologies into clinical severity tiers and reports faithfulness + false-positive behavior per tier.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

GDRIVE_ROOT = '/content/drive/MyDrive/cxr_faithfulness'
exec(open(f'{GDRIVE_ROOT}/config/startup.py').read())


Mounted at /content/drive
⏳ Installing strictly pinned architecture packages onto Colab's native stack (~30s)...
✅ Packages ready! Using native modern PyTorch and NumPy.


In [2]:
import json, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, kruskal
from statsmodels.stats.multitest import multipletests
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ROOT         = Path(GDRIVE_ROOT)
RESULTS_PATH = ROOT / 'results'
DATA_PATH    = ROOT / 'data' / 'processed'
FIGPATH      = ROOT / 'figures'
FIGPATH.mkdir(parents=True, exist_ok=True)

MODEL_NAMES  = ['densenet121', 'convnextv2_tiny', 'swinb_lora']
DISPLAY_NAMES= {'densenet121':'DenseNet121','convnextv2_tiny':'ConvNeXtV2-Tiny','swinb_lora':'Swin-B LoRA'}
MIN_TEST_N   = 30
ALPHA        = 0.05
BOOTSTRAP_N  = 1000
SEED         = 42
np.random.seed(SEED)
print("✓ NB08 setup ready")


✓ NB08 setup ready


CRITICAL data caveat:
Faithfulness CSVs contain only ~10 of 14 classes. `Pneumothorax`, `Consolidation`, `Atelectasis`, `Calcification` have NO mIoU/monotonicity data (never a top-predicted class in NB04). Therefore the **Critical tier reduces to Aortic enlargement** and **Moderate to Cardiomegaly + Pleural effusion** for faithfulness. FP-count tables DO cover all 14 classes. NB08 must report which classes are present vs absent per tier, and must not crash on empty groups.


In [3]:
SEVERITY_TIERS = {
    'Aortic enlargement': 'Critical',
    'Pneumothorax':       'Critical',
    'Pleural effusion':   'Moderate',
    'Consolidation':      'Moderate',
    'Atelectasis':        'Moderate',
    'Cardiomegaly':       'Moderate',
    'Calcification':      'Mild',
    'ILD':                'Mild',
    'Infiltration':       'Mild',
    'Lung Opacity':       'Mild',
    'Nodule/Mass':        'Mild',
    'Other lesion':       'Mild',
    'Pleural thickening': 'Mild',
    'Pulmonary fibrosis': 'Mild',
}
TIER_ORDER = ['Critical', 'Moderate', 'Mild']


In [4]:
miou   = pd.read_csv(RESULTS_PATH / 'miou_results.csv')
mono   = pd.read_csv(RESULTS_PATH / 'monotonicity_results.csv')
lime   = pd.read_csv(RESULTS_PATH / 'lime_ig_agreement.csv')
delins = pd.read_csv(RESULTS_PATH / 'deletion_insertion.csv')
fleiss = pd.read_csv(DATA_PATH / 'fleiss_kappa.csv')

key = ['image_id', 'model', 'target_class']

# LIME: keep only valid + patho rows (no GT box on healthy)
lime_valid = lime[(lime['iou_valid'] == True) & (lime['subset'] == 'patho')]

merged = (
    miou[miou['subset'] == 'patho'][key + ['miou']]
    .merge(mono[key + ['monotonicity_auc']], on=key, how='inner')
    .merge(lime_valid[key + ['lime_ig_iou']], on=key, how='left')   # left: LIME may be NaN
    .merge(delins[key + ['deletion_auc']], on=key, how='left')
)
merged['tier'] = merged['target_class'].map(SEVERITY_TIERS)
assert merged['tier'].notna().all(), \
    f"Unmapped classes: {merged.loc[merged['tier'].isna(),'target_class'].unique()}"
print(merged.groupby('tier')['image_id'].count())
print("Present classes per tier:")
print(merged.groupby('tier')['target_class'].unique())

expected_classes = set(SEVERITY_TIERS.keys())
present_classes = set(merged['target_class'].unique())
absent_classes = expected_classes - present_classes
print(f"\nAbsent classes (no top-1 predictions in NB04): {absent_classes}")

tier
Critical    533
Mild        218
Moderate    257
Name: image_id, dtype: int64
Present classes per tier:
tier
Critical                                 [Aortic enlargement]
Mild        [Pleural thickening, Lung Opacity, Pulmonary f...
Moderate                     [Pleural effusion, Cardiomegaly]
Name: target_class, dtype: object

Absent classes (no top-1 predictions in NB04): {'Atelectasis', 'Pneumothorax', 'Consolidation', 'Calcification'}


In [5]:
def bootstrap_ci(values, n_boot=BOOTSTRAP_N, ci=0.95, seed=None):
    vals = np.asarray(values, dtype=float)
    vals = vals[~np.isnan(vals)]
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    boots = np.array([vals[rng.integers(0, len(vals), len(vals))].mean()
                      for _ in range(n_boot)])
    return float(vals.mean()), float(np.quantile(boots, 0.025)), float(np.quantile(boots, 0.975))

def stable_seed(text):
    return int(hashlib.md5(text.encode()).hexdigest(), 16) % 100000


In [6]:
kappa_map = dict(zip(fleiss['class_name'], fleiss['fleiss_kappa']))
rows = []
for model in MODEL_NAMES:
    for tier in TIER_ORDER:
        sub = merged[(merged['model'] == model) & (merged['tier'] == tier)]
        if sub.empty:
            continue
        sd = SEED + stable_seed(f'{model}:{tier}')
        m_mean, m_lo, m_hi = bootstrap_ci(sub['miou'], seed=sd+1)
        mo_mean, mo_lo, mo_hi = bootstrap_ci(sub['monotonicity_auc'], seed=sd+2)
        l_mean, l_lo, l_hi = bootstrap_ci(sub['lime_ig_iou'], seed=sd+3)
        d_mean, d_lo, d_hi = bootstrap_ci(sub['deletion_auc'], seed=sd+4)
        n = sub['image_id'].nunique()
        tier_classes = sub['target_class'].unique().tolist()
        kappa_tier = np.nanmean([kappa_map.get(c, np.nan) for c in tier_classes])
        rows.append(dict(
            model=model, tier=tier, n=n,
            n_classes=len(tier_classes),
            mean_kappa=round(kappa_tier, 4),
            miou_mean=round(m_mean,4), miou_ci_lo=round(m_lo,4), miou_ci_hi=round(m_hi,4),
            mono_mean=round(mo_mean,4), mono_ci_lo=round(mo_lo,4), mono_ci_hi=round(mo_hi,4),
            lime_mean=round(l_mean,4), lime_ci_lo=round(l_lo,4), lime_ci_hi=round(l_hi,4),
            delins_mean=round(d_mean,4),
            flag='†' if n < MIN_TEST_N else '',
        ))
severity_df = pd.DataFrame(rows)
severity_df.to_csv(RESULTS_PATH / 'severity_analysis.csv', index=False)
print(severity_df.to_string(index=False))


          model     tier   n  n_classes  mean_kappa  miou_mean  miou_ci_lo  miou_ci_hi  mono_mean  mono_ci_lo  mono_ci_hi  lime_mean  lime_ci_lo  lime_ci_hi  delins_mean flag
    densenet121 Critical 174          1      0.7802     0.0373      0.0327      0.0421     0.3640      0.3535      0.3740     0.0976      0.0898      0.1052       0.7887     
    densenet121 Moderate  84          2      0.7470     0.1590      0.1475      0.1710     0.2129      0.1874      0.2351     0.0945      0.0826      0.1072       0.5869     
    densenet121     Mild  66          6      0.4837     0.0644      0.0509      0.0783     0.1294      0.1123      0.1473     0.0667      0.0577      0.0760       0.2931     
convnextv2_tiny Critical 168          1      0.7802     0.0269      0.0244      0.0293     0.6555      0.6332      0.6760     0.1470      0.1381      0.1559       0.5682     
convnextv2_tiny Moderate  87          2      0.7470     0.1765      0.1626      0.1918     0.7544      0.7161      0.7879    

In [7]:
def paired_wilcoxon_by_tier(df, metric, min_n=15):
    out = []
    for tier in TIER_ORDER:
        sub = df[df['tier'] == tier]
        piv = sub.pivot_table(index=['image_id','target_class'], columns='model', values=metric)
        present = [m for m in MODEL_NAMES if m in piv.columns]
        for i in range(len(present)):
            for j in range(i+1, len(present)):
                a, b = present[i], present[j]
                pair = piv[[a,b]].dropna()
                if len(pair) < min_n:
                    continue
                diff = (pair[a] - pair[b]).values
                if np.all(diff == 0) or np.unique(diff).size < 2:
                    continue
                stat, p = wilcoxon(pair[a], pair[b])
                out.append(dict(tier=tier, metric=metric, pair=f'{a}_vs_{b}',
                                n=len(pair), statistic=float(stat), p_raw=float(p)))
    return out

all_tests = []
for metric in ['miou', 'monotonicity_auc', 'lime_ig_iou', 'deletion_auc']:
    all_tests += paired_wilcoxon_by_tier(merged, metric)

wilcox_df = pd.DataFrame(all_tests)
if not wilcox_df.empty:
    rej, p_bonf, _, _ = multipletests(wilcox_df['p_raw'], alpha=ALPHA, method='bonferroni')
    wilcox_df['p_bonf'] = p_bonf
    wilcox_df['significant'] = rej
wilcox_df.to_csv(RESULTS_PATH / 'severity_wilcoxon.csv', index=False)
print(wilcox_df.to_string(index=False))


    tier           metric                           pair   n  statistic        p_raw       p_bonf  significant
Critical             miou densenet121_vs_convnextv2_tiny 135     3423.0 1.037599e-02 3.735356e-01        False
Critical             miou      densenet121_vs_swinb_lora 156        9.0 2.828996e-27 1.018439e-25         True
Critical             miou  convnextv2_tiny_vs_swinb_lora 150        0.0 2.299606e-26 8.278581e-25         True
Moderate             miou densenet121_vs_convnextv2_tiny  51      439.0 3.575906e-02 1.000000e+00        False
Moderate             miou      densenet121_vs_swinb_lora  49       69.0 9.573675e-10 3.446523e-08         True
Moderate             miou  convnextv2_tiny_vs_swinb_lora  58      521.0 9.602741e-03 3.456987e-01        False
    Mild             miou densenet121_vs_convnextv2_tiny  33       25.0 2.104789e-07 7.577240e-06         True
    Mild             miou      densenet121_vs_swinb_lora  34        1.0 2.328306e-10 8.381903e-09         True
 

In [8]:
# (a) FP counts per tier (covers all 14 classes)
fp_rows = []
for model in MODEL_NAMES:
    fp = pd.read_csv(RESULTS_PATH / f'{model}_fp_analysis.csv')
    fp['tier'] = fp['class'].map(SEVERITY_TIERS)
    g = fp.groupby('tier').agg(
        type_a_fp=('type_a_hallucination_fp','sum'),
        type_b_fp=('type_b_wrong_class_fp','sum'),
        n_classes=('class','count')).reset_index()
    g['model'] = model
    fp_rows.append(g)
fp_tier_df = pd.concat(fp_rows, ignore_index=True)
fp_tier_df.to_csv(RESULTS_PATH / 'severity_fp_counts.csv', index=False)
print(fp_tier_df.to_string(index=False))

# (b) Type B spatial anchoring (mass-in-box delta) per tier
tb = pd.read_csv(RESULTS_PATH / 'fp_taxonomy_typeB.csv')
tb = tb[tb['mass_in_box'].notna()].copy()
tb['tier'] = tb['target_class'].map(SEVERITY_TIERS)
tb['delta'] = tb['mass_in_box'] - tb['null_mass_in_box']
anchor_rows = []
for tier in TIER_ORDER:
    sub = tb[tb['tier'] == tier]
    if len(sub) < 5:
        continue
    mean, lo, hi = bootstrap_ci(sub['delta'], seed=SEED+stable_seed(tier))
    try:
        _, p = wilcoxon(sub['delta'], alternative='greater')
    except ValueError:
        p = np.nan
    anchor_rows.append(dict(tier=tier, n=len(sub),
                            mean_delta=round(mean,4), ci_lo=round(lo,4), ci_hi=round(hi,4),
                            wilcoxon_p=p))
anchor_df = pd.DataFrame(anchor_rows)
anchor_df.to_csv(RESULTS_PATH / 'severity_fp_anchoring.csv', index=False)
print(anchor_df.to_string(index=False))


    tier  type_a_fp  type_b_fp  n_classes           model
Critical         31         67          2     densenet121
    Mild        105        744          8     densenet121
Moderate         14        131          4     densenet121
Critical         29         65          2 convnextv2_tiny
    Mild         76        542          8 convnextv2_tiny
Moderate         18        139          4 convnextv2_tiny
Critical         13         42          2      swinb_lora
    Mild         82        531          8      swinb_lora
Moderate          7         94          4      swinb_lora
    tier  n  mean_delta  ci_lo  ci_hi   wilcoxon_p
Critical 52      0.1149 0.0792 0.1550 2.159422e-07
Moderate 24      0.1428 0.0774 0.2179 1.611710e-04
    Mild 89      0.0762 0.0565 0.0993 6.321803e-08


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metrics = [('miou_mean','miou_ci_lo','miou_ci_hi','Mean mIoU (PRIMARY)'),
           ('mono_mean','mono_ci_lo','mono_ci_hi','Mean Monotonicity AUC (SECONDARY)')]
colors = {'densenet121':'#01696f','convnextv2_tiny':'#da7101','swinb_lora':'#7a39bb'}
x = np.arange(len(TIER_ORDER)); w = 0.25
for ax, (mcol, lo, hi, title) in zip(axes, metrics):
    for k, model in enumerate(MODEL_NAMES):
        vals, elo, ehi = [], [], []
        for tier in TIER_ORDER:
            r = severity_df[(severity_df['model']==model)&(severity_df['tier']==tier)]
            if r.empty:
                vals.append(0); elo.append(0); ehi.append(0); continue
            r = r.iloc[0]
            vals.append(r[mcol]); elo.append(r[mcol]-r[lo]); ehi.append(r[hi]-r[mcol])
        ax.bar(x + (k-1)*w, vals, w, yerr=[elo,ehi], capsize=4,
               color=colors[model], label=DISPLAY_NAMES[model], alpha=0.9)
    ax.set_xticks(x); ax.set_xticklabels(TIER_ORDER)
    ax.set_title(title); ax.legend(fontsize=8)
    ax.spines[['top','right']].set_visible(False)
fig.suptitle('Figure 5 — Faithfulness by Clinical Severity Tier', fontsize=13)
plt.tight_layout()
fig.savefig(FIGPATH / 'figure5_severity_barplot.png', dpi=300, bbox_inches='tight')
plt.close(fig)
print("✓ figure5_severity_barplot.png saved")


✓ figure5_severity_barplot.png saved


In [10]:
TAB_PATH = RESULTS_PATH / 'paper_tables'
TAB_PATH.mkdir(parents=True, exist_ok=True)

# Format rows for presentation
table_rows = []
for tier in TIER_ORDER:
    for model in MODEL_NAMES:
        r_df = severity_df[(severity_df['model'] == model) & (severity_df['tier'] == tier)]
        if r_df.empty:
            continue
        r = r_df.iloc[0]
        table_rows.append({
            'Clinical Tier': tier,
            'Model': DISPLAY_NAMES[model],
            'Support (n)': f"{r['n']} {r['flag']}",
            'Reader $\\kappa$': f"{r['mean_kappa']:.3f}" if pd.notna(r['mean_kappa']) else 'N/A',
            'mIoU [95% CI]': f"{r['miou_mean']:.3f} [{r['miou_ci_lo']:.3f}, {r['miou_ci_hi']:.3f}]",
            'Mono. AUC': f"{r['mono_mean']:.3f}",
            'LIME-IG IoU': f"{r['lime_mean']:.3f}",
            'Del. AUC': f"{r['delins_mean']:.3f}"
        })

table4_df = pd.DataFrame(table_rows)
table4_df.to_csv(TAB_PATH / 'table4_severity.csv', index=False)

# LaTeX export
caption = (
    "Faithfulness metrics aggregated by clinical severity tier. "
    "Note that the Critical tier is exclusively represented by Aortic enlargement, "
    "as Pneumothorax, Consolidation, Atelectasis, and Calcification cases were never "
    "the top-1 predictions for any model, and thus lack faithfulness maps."
)

tex = table4_df.to_latex(index=False, escape=False, caption=caption, label='tab:severity')
with open(TAB_PATH / 'table4_severity.tex', 'w') as f:
    f.write(tex)

print("✓ Table 4 saved (CSV + LaTeX)")


✓ Table 4 saved (CSV + LaTeX)


In [11]:
expected = ['severity_analysis.csv','severity_wilcoxon.csv',
            'severity_fp_counts.csv','severity_fp_anchoring.csv']
ok = True
for f in expected:
    e = (RESULTS_PATH / f).exists()
    print(f"  {'✓' if e else '✗'} {f}")
    ok &= e
print(f"  {'✓' if (FIGPATH/'figure5_severity_barplot.png').exists() else '✗'} figure5_severity_barplot.png")

# Non-degenerate checks
assert severity_df['tier'].nunique() >= 2, "Severity collapsed to one tier"
assert severity_df['miou_mean'].nunique() > 1, "mIoU identical across all tiers — check tier mapping"
print("✓ NB08 complete." if ok else "✗ NB08 missing outputs")


  ✓ severity_analysis.csv
  ✓ severity_wilcoxon.csv
  ✓ severity_fp_counts.csv
  ✓ severity_fp_anchoring.csv
  ✓ figure5_severity_barplot.png
✓ NB08 complete.
